# インポート

In [1]:
import asyncio
import os
from rich import print
from pprint import pprint
from strands import Agent
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from typing import AsyncIterator, Any, Dict, Any,Union, Optional, Callable
from collections.abc import AsyncIterator
from strands import tool
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from mcp import stdio_client, StdioServerParameters
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
async def send_event(
    queue: asyncio.Queue | None, 
    message: str, 
    stage: str, 
    tool_name: str | None = None) -> None:
    """サブエージェントのステータスを送信"""
    """サブエージェントの進捗状況をキューに送信する。
    
    サブエージェントの実行ステータス（開始、完了、ツール使用など）を
    イベントとしてキューに送信します。キューがNoneの場合は何も行いません。
    
    Args:
        queue: イベントを送信するasyncio.Queue。Noneの場合は送信しない。
        message: 進捗状況を示すメッセージ（例：「サブエージェント「AWSマスター」が呼び出されました」）
        stage: 実行ステージ（例：「start」「complete」「tool_use」）
        tool_name: ツール名（オプショナル）。ツール使用時に指定する。
    
    Returns:
        None
    
    Example:
        >>> queue = asyncio.Queue()
        >>> await send_event(queue, "処理を開始しました", "start")
        >>> await send_event(queue, "ツールを実行中", "tool_use", "aws_master")
    """
    if not queue:
        return 

    progress = {"message": message, "stage": stage}
    if tool_name:
        progress["tool_name"] = tool_name

    await queue.put(
        {
            "event": {
                "subAgentProgress": progress
            }
        }
    )

In [3]:
queue = asyncio.Queue()
await send_event(queue, "処理を開始しました", "start")
await send_event(queue, "ツールを実行中", "tool_use", "aws_master")

pprint(queue)
pprint(queue._queue)


<Queue at 0x7404422437d0 maxsize=0 _queue=[{'event': {'subAgentProgress': {'message': '処理を開始しました', 'stage': 'start'}}}, {'event': {'subAgentProgress': {'message': 'ツールを実行中', 'stage': 'tool_use', 'tool_name': 'aws_master'}}}] tasks=2>
deque([{'event': {'subAgentProgress': {'message': '処理を開始しました',
                                       'stage': 'start'}}},
       {'event': {'subAgentProgress': {'message': 'ツールを実行中',
                                       'stage': 'tool_use',
                                       'tool_name': 'aws_master'}}}])


In [4]:
async def merge_streams(
    stream: AsyncIterator[Dict[str, Any]], 
    queue: asyncio.Queue
) -> AsyncIterator[Dict[str, Any]]:
    """親子エージェントのストリームを統合する。
    
    監督者エージェント（メインストリーム）とサブエージェント（キュー）から
    来るイベントを統合し、到着順にyieldします。両方のストリームを並行して
    待機し、どちらかが先に到着したら即座に処理します。
    
    Args:
        stream: 監督者エージェントからの非同期イテレータ（ストリーム）。
                各イベントは辞書形式で、`{"event": {...}}`の構造を持つ。
        queue: サブエージェントからのイベントを格納するasyncio.Queue。
               各イベントは辞書形式で、`{"event": {...}}`の構造を持つ。
    
    Yields:
        Dict[str, Any]: 統合されたイベント。メインストリームまたはキューから
                        到着したイベントを到着順に返す。
    
    Note:
        - ストリームが終了（Noneを返す）し、キューが空になったら終了します。
        - キューからの取得でエラーが発生した場合、そのタスクは無視されます。
    
    Example:
        >>> async def main_stream():
        ...     yield {"event": {"contentBlockDelta": {"delta": {"text": "Hello"}}}}
        ...     yield None
        >>> queue = asyncio.Queue()
        >>> await queue.put({"event": {"subAgentProgress": {"message": "開始", "stage": "start"}}})
        >>> async for event in merge_streams(main_stream(), queue):
        ...     print(event)
    """
    create_task = asyncio.create_task
    
    main = create_task(anext(stream, None))
    sub = create_task(queue.get())
    waiting = {main, sub}

    while waiting:
        ready_chunks, waiting = await asyncio.wait(
            waiting, return_when=asyncio.FIRST_COMPLETED
        )
        for ready_chunk in ready_chunks:
            # 監督者エージェントのチャンクを処理
            if ready_chunk == main:
                event = ready_chunk.result()
                if event is not None:
                    yield event
                    main = create_task(anext(stream, None))
                    waiting.add(main)
                else:
                    main = None

            # サブエージェントのチャンク   を処理
            elif ready_chunk == sub:
                try:
                    sub_event = ready_chunk.result()
                    yield sub_event
                    sub = create_task(queue.get())
                    waiting.add(sub)
                except Exception:
                    sub = None

        if main is None and queue.empty():
                break


In [5]:
# merge_streamsの実験

async def main_stream():
    yield {"event": {"contentBlockDelta": {"delta": {"text": "Hello"}}}}
    yield {"event": {"contentBlockDelta": {"delta": {"text": "GoodAfternoon"}}}}
    yield {"event": {"contentBlockDelta": {"delta": {"text": "GoodEvening"}}}}
    
queue = asyncio.Queue()
await queue.put({"event": {"subAgentProgress": {"message": "開始", "stage": "start"}}})
await queue.put({"event": {"subAgentProgress": {"message": "ツールを実行中", "stage": "tool_use", "tool_name": "aws_master"}}})
await queue.put({"event": {"subAgentProgress": {"message": "APIを実行中", "stage": "tool_use", "tool_name": "aws_api"}}})

async for event in merge_streams(main_stream(), queue):
    print()

In [6]:
async def extract(
    queue: Optional[asyncio.Queue],
    agent: str,
    event: Union[str, Dict[str, Any]],
    state: Dict[str, str]
) -> None:
    """「ストリーミングイベント」から内容を抽出し、キューに送信する。
    
    エージェントのストリームから来るイベントを処理し、以下の操作を行います：
    1. テキストイベントを検出して`state["text"]`に蓄積
    2. ツール使用イベントを検出して進捗を通知
    3. イベントをキューに送信してリアルタイム表示を可能にする
    
    Args:
        queue: イベントを送信するasyncio.Queue。Noneの場合は送信しない。
        agent: エージェント名（例：「AWSマスター」）。進捗メッセージに使用される。
        event: ストリームから来るイベント。文字列または辞書形式。
              - 文字列の場合：直接テキストとして処理
              - 辞書の場合：`{"event": {...}}`の構造を持つ
        state: テキストを蓄積する辞書。`{"text": ""}`の形式。
              `state["text"]`にテキストが追加される（参照渡し）。
    
    Returns:
        None
    
    Note:
        - `state`は参照渡しのため、この関数内での変更が呼び出し元に反映される
        - キューがNoneの場合、イベントは送信されないが、`state`への蓄積は行われる
        - ツール使用が検出された場合、`send_event`で進捗が通知される
    
    Example:
        >>> queue = asyncio.Queue()
        >>> state = {"text": ""}
        >>> await extract(queue, "AWSマスター", "Hello", state)
        >>> print(state["text"])  # "Hello"
    """
    if isinstance(event, str):
        # stateは存在しなくてもいいと思うのでコメントアウトしてみる。
        # state["text"] += event
        if queue:
            delta = {"delta": {"text": event}}
            await queue.put(
                {"event": {"contentBlockDelta": delta}}
            )
    elif isinstance(event, dict) and "event" in event:
        event_data = event["event"]

        # ツール使用を検出
        if "contentBlockStart" in event_data:
            block = event_data["contentBlockStart"]
            start_data = block.get("start", {})
            if "toolUse" in start_data:
                tool_use = start_data["toolUse"]
                tool = tool_use.get("name", "unknown")
                await send_event(
                    queue=queue,
                    message=f"「{agent}」がツール「{tool}」を実行中",
                    stage="tool_use",
                    tool_name=tool
                )
        # 以下、なんの意味もない処理だと思うのでコメントアウトしてみる。
        # elif "contentBlockDelta" in event_data:
        #     block = event_data["contentBlockDelta"]
        #     delta = block.get("delta", {})
        #     if "text" in delta:
        #         state["text"] += delta["text"]
        
        if queue:
            await queue.put(event)

In [7]:
queue = asyncio.Queue()
state = {"text": ""}
await extract(queue, "AWSマスター", "Hello", state)
await extract(queue, "hogehoge", "GoodAfternoon", state)
print(queue) 
print(queue._queue) # "Hello"

Task was destroyed but it is pending!
task: <Task pending name='Task-48' coro=<Queue.get() done, defined at /home/ryoyamasuda/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/queues.py:149> wait_for=<Future cancelled>>


<Queue maxsize=0 _queue=[{'event': {'contentBlockDelta': {'delta': {'text': 'Hello'}}}}, {'event': 
{'contentBlockDelta': {'delta': {'text': 'GoodAfternoon'}}}}] tasks=2>

deque([
    {'event': {'contentBlockDelta': {'delta': {'text': 'Hello'}}}},
    {'event': {'contentBlockDelta': {'delta': {'text': 'GoodAfternoon'}}}}
])

In [8]:
# send_event: 進捗通知を送るだけ
# extract: ストリームイベントを処理し、必要に応じてsend_eventを呼ぶ

async def invoke(
    agent: str, 
    query: str, 
    mcp: MCPClient, 
    create_agent: Callable[[], Agent], 
    queue: asyncio.Queue
) -> str:
    """サブエージェントを呼び出し、ストリーミング処理を実行する。
    
    MCPクライアントを使用してサブエージェントを起動し、クエリに対する
    ストリーミング応答を処理します。処理中は進捗状況をキューに送信し、
    最終的なテキスト結果を返します。
    
    Args:
        agent: エージェント名（例：「AWSマスター」）。進捗メッセージに使用される。
        query: エージェントに送信するクエリ（質問や指示）。
        mcp: MCPクライアント。コンテキストマネージャーとして使用され、
             エージェント実行中にMCP接続を管理する。
        create_agent: エージェントインスタンスを作成する関数。
                     引数なしで呼び出し、Agentオブジェクトを返す。
        queue: イベントを送信するasyncio.Queue。Noneの場合は進捗通知を送信しない。
    
    Returns:
        str: エージェントが生成した完全なテキスト応答。
             エラーが発生した場合はエラーメッセージを返す。
    
    Note:
        - MCPクライアントは`with mcp:`ブロック内で管理され、
          処理完了時に自動的にクリーンアップされる。
        - 処理開始時と完了時に進捗イベントをキューに送信する。
        - ストリームから来る各イベントは`extract`関数で処理される。
    
    Example:
        >>> queue = asyncio.Queue()
        >>> mcp = MCPClient(...)
        >>> result = await invoke(
        ...     "AWSマスター",
        ...     "S3について教えて",
        ...     mcp,
        ...     lambda: Agent(...),
        ...     queue
        ... )
        >>> print(result)  # エージェントの応答テキスト
    """
    state = {"text": ""}
    await send_event(
        queue=queue, 
        message=f"サブエージェント「{agent}」が呼び出されました", 
        stage="start"
    )

    try:
        # MCPを起動
        with mcp:
            # エージェントを作成
            agent_obj = create_agent()
            # この時点でMCPは起動済み。エージェントがツールを使うと、MCP経由で実行される
            async for event in agent_obj.stream_async(query):
                # ストリーミングを受取り、キューに送信
                await extract(
                    queue=queue, 
                    agent=agent, 
                    event=event, 
                    state=state
                )
        # エージェントの対応が完了したことを通知(キューに追加)
        await send_event(
            queue=queue,
            message=f"「{agent}」が対応を完了しました",
            stage="complete"
        )
        return state["text"]
    
    except Exception:
        return f"{agent}エージェントの処理に失敗しました"

    

In [9]:
# モックのMCPクライアント
class MockMCPClient:
    def __enter__(self):
        return self
    def __exit__(self, *args):
        return False

# モックのAgent
class MockAgent:
    async def stream_async(self, query):
        yield {"event": {"contentBlockDelta": {"delta": {"text": "Hello"}}}}
        yield {"event": {"contentBlockDelta": {"delta": {"text": " World"}}}}

# 実行
async def simple_test():
    queue = asyncio.Queue()
    mcp = MockMCPClient()
    
    result = await invoke(
        agent="テスト",
        query="質問",
        mcp=mcp,
        create_agent=lambda: MockAgent(),
        queue=queue
    )
    
    print(f"結果: {result}")

await simple_test()

結果:

In [10]:
class AwsMasterState:
    def __init__(self):
        self.client = None
        self.queue = None


In [11]:

_state = AwsMasterState()


In [12]:

def setup_aws_master(
    queue: asyncio.Queue
    ) -> None:
    _state.queue = queue
    if queue and not _state.client:
        try:
            _state.client = MCPClient(
                transport_callable = lambda: streamablehttp_client(
                    "https://knowledge-mcp.global.api.aws"
                )
                # lambda: streamablehttp_client(
                #     "https://knowledge-mcp.global.api.aws"
                # )
            )
        except Exception:
            _state.client = None


In [13]:

def _create_agent():
    """サブエージェントを作成"""
    if not _state.client:
        return None
    return Agent(
        model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        tools=_state.client.list_tools_sync()
    )


In [14]:

@tool
async def aws_master(query):
    """AWSマスターエージェント"""
    if not _state.client:
        return "MCPクライアントが利用不可です"
    return await invoke(
        agent="AWS マスター", 
        query=query, 
        mcp=_state.client,
        create_agent=_create_agent,
        queue=_state.queue
    )

In [15]:
class ApiMasterState:
    def __init__(self):
        self.client = None
        self.queue = None

In [16]:
_state1 = ApiMasterState()

In [17]:
def setup_api_master(queue):
    """新規キューを受け取り、MCPクライアントを準備"""
    _state1.queue = queue
    if queue and not _state1.client:
        try:
            _state1.client = MCPClient(
                lambda: stdio_client(StdioServerParameters(
                    command="uvx",
                    args=["awslabs.aws-api-mcp-server"],
                    env=os.environ.copy()
                ))
            )
        except Exception:
            _state1.client = None

In [18]:
def _create_agent1():
    """サブエージェントを作成"""
    if not _state1.client:
        return None
    return Agent(
        model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        tools=_state1.client.list_tools_sync()
    )

In [19]:
@tool
async def api_master(query):
    """APIマスターエージェント"""
    if not _state1.client:
        return "MCPクライアントが利用不可です"
    return await invoke(
        "APIマスター", query, _state1.client,
        _create_agent1, _state.queue
    )

In [20]:
def _create_orchestrator():
    """監督者エージェントを作成"""
    return Agent(
        model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        tools=[aws_master, api_master],
        system_prompt="""2体のサブエージェントを使って日本語で応対して。
1. AWSマスター：AWSドキュメントなどを参照できます。
2. APIマスター：AWSアカウントをAPIで操作できます。"""
    )

In [21]:

# アプリケーションを初期化
app = BedrockAgentCoreApp()
orchestrator = _create_orchestrator()

@app.entrypoint
async def invoke1(payload):
    """呼び出し処理の開始地点"""
    prompt = payload.get("input", {}).get("prompt", "")
    
    # サブエージェント用のキューを初期化
    queue = asyncio.Queue()
    setup_aws_master(queue)
    setup_api_master(queue)
    
    try:
        # 監督者エージェントを呼び出し、ストリームを統合
        stream = orchestrator.stream_async(prompt)
        async for event in merge_streams(stream, queue):
            yield event
            
    finally:
        # キューをクリーンアップ
        setup_aws_master(None)
        setup_api_master(None)

# APIサーバーを起動
# if __name__ == "__main__":
#     app.run()

In [24]:
# ストリーミングのテキスト回答のみを表示
import sys

async def test_invoke1_text_only():
    """テキストのみを表示する関数（ストリーミング出力）"""
    payload = {
        "input": {
            "prompt": "AWSのS3について教えてください"
        }
    }
    
    async for event in invoke1(payload):
        if "event" in event:
            event_data = event["event"]
            
            # テキストデルタのみを抽出して表示
            if "contentBlockDelta" in event_data:
                delta = event_data["contentBlockDelta"].get("delta", {})
                if "text" in delta:
                    # sys.stdout.writeを使って確実にストリーミング出力
                    sys.stdout.write(delta["text"])
                    sys.stdout.flush()

# 実行
await test_invoke1_text_only()


AWSのSAWSのS3（Simple Storage Service）について詳しい3（Simple Storage Service）について詳しい情報をお届けします。も情報をお届けします。もう少し詳細な情う少し詳細な情報をAWSマス報をAWSマスターから取得してターから取得してみましょう。みましょう。
Tool #3: aws_master


Failed to detach context
Traceback (most recent call last):
  File "/home/ryoyamasuda/Documents/git_repo_RyoyaMasuda/agent-book/chapter6_self/.venv/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/ryoyamasuda/Documents/git_repo_RyoyaMasuda/agent-book/chapter6_self/.venv/lib/python3.12/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self._current_context.reset(token)
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x740458ec6b60> at 0x74042f872b00> was created in a different Context


Amazon S3についてAmazon S3について詳しく調べます詳しく調べますね。基本概念、特徴、ね。基本概念、特徴、ユースケース、料金体ユースケース、料金体系、運用方法、最系、運用方法、最新機能といった幅広いト新機能といった幅広いトピックについて情報を収ピックについて情報を収集します。

まずは集します。

まずはAmazon S3の基本的Amazon S3の基本的な情報を検索しましょう。な情報を検索しましょう。
Tool #1: aws___search_documentation
SS3の料金体系と3の料金体系と最新機能について、もう最新機能について、もう少し詳しい情少し詳しい情報を集めましょう。報を集めましょう。
Tool #2: aws___search_documentation
次次に運用方法について詳しいに運用方法について詳しい情報を得ましょう。情報を得ましょう。
Tool #3: aws___search_documentation
SS3の3の最新機能につい最新機能についても確認してみても確認してみましょう。ましょう。
Tool #4: aws___search_documentation
それそれでは、では、詳しい情報を詳しい情報を得るために、得るために、いくつかのいくつかのAWS公AWS公式ドキュ式ドキュメントをさらに読メントをさらに読み込みましみ込みましょう。ょう。
Tool #5: aws___read_documentation
SS3のドキュメント3のドキュメントの続の続きを読きを読みましょう。みましょう。
Tool #6: aws___read_documentation
続続きをきを読み読みましょう。ましょう。
Tool #7: aws___read_documentation
SS3のドキュメント3のドキュメントの残りの部分を読みの残りの部分を読みましょう。ましょう。
Tool #8: aws___read_documentation


Task was destroyed but it is pending!
task: <Task pending name='Task-755' coro=<Queue.get() running at /home/ryoyamasuda/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/queues.py:158> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-1894' coro=<Queue.get() running at /home/ryoyamasuda/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/queues.py:158> wait_for=<Future pending cb=[Task.task_wakeup()]>>


最最後の後の部分も部分も確認し確認しましょう。ましょう。
Tool #9: aws___read_documentation
最後に、S最後に、S3の料金体3の料金体系について詳細を確系について詳細を確認しましょう。認しましょう。
Tool #10: aws___read_documentation
集集めためた情報をもとに、情報をもとに、Amazon S3の基Amazon S3の基本概本概念、念、特徴、特徴、ユースケースユースケース、料、料金体金体系、運系、運用方用方法、最法、最新機新機能について能について詳しく説明します。詳しく説明します。

# Amazon

# Amazon S3（ S3（Simple Storage Service）のSimple Storage Service）の総合総合ガイドガイド

## 1. 基

## 1. 基本概念

Amazon S本概念

Amazon S3は、AWS3は、AWS（Amazon Web Services）（Amazon Web Services）が提供するが提供するオブオブジェクトジェクトストレージサストレージサービスですービスです。以下が。以下が基基本的本的な概念ですな概念です：

### ：

### 1.1 1.1 オブジェオブジェクトストレクトストレージとージとは
Amazonは
Amazon S3は S3はオオブジェクブジェクトストトストレージサレージサービスであービスであり、データをり、データをオオブジェクブジェクトとして保トとして保存します存します。各。各オブジェオブジェクトはクトはデデータ本ータ本体と体とそそれをれを説説明するメ明するメタデータからタデータから構成されています構成されています。

### 。

### 1.2 1.2 基本要基本要素素

#### バ

#### バケットケット
- S
- S3の3の基本的基本的なコなコンテンテナであナでありり、す、すべてのオべてのオブジェクブジェクトはトはババケットケット内に保内に保存されます存されます
- バケット
- バケット名は名はAAWWS全S全体で一意体で一意である必である必要があります要があります
- S3には
- S3には以下の以下の4種4種類のバケ類のバケットタットタイプがあイプがあります：
  - **ります：
  - *

Failed to detach context
Traceback (most recent call last):
  File "/home/ryoyamasuda/Documents/git_repo_RyoyaMasuda/agent-book/chapter6_self/.venv/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/ryoyamasuda/Documents/git_repo_RyoyaMasuda/agent-book/chapter6_self/.venv/lib/python3.12/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self._current_context.reset(token)
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x740458ec6b60> at 0x74042d77ed00> was created in a different Context
Failed to detach context
Traceback (most recent call last):
  File "/home/ryoyamasuda/Documents/git_repo_RyoyaMasuda/agent-book/chapter6_self/.venv/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/home/ryoyamasuda/Documents/git_repo_RyoyaMasuda/agent-book/chapter6_self/.venv/lib/python3.

勧めします。